## Visualization of the TTF artifacts and modifications

In [ ]:
import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '2'

from typing import Callable, Optional, Union, Sequence
from tqdm.notebook import tqdm
import numpy as np
import torch
from nflows.transforms import Transform
import normflows as nf
import matplotlib.pyplot as plt
import openturns as ot

from tailnflows.targets.synth_targets import GaussianCopula
from tailnflows.models.extreme_transformations import TailAffineMarginalTransform, ModifiedTailAffineMarginalTransform

torch.cuda.set_device(3)
device = torch.device(f"cuda:{3}" if torch.cuda.is_available() else "cpu")

print("Using device: ", device)
print("Current CUDA device:", torch.cuda.current_device())
print("All CUDA devices:", torch.cuda.device_count())

DEFAULT_DTYPE = torch.float32

### Prepare target distribution, base distribution and visualization parameters that all models share:

In [ ]:
# Mixed-tailed Gaussian Copula target
df = 2.5
target = GaussianCopula(
    num_light = 1,
    num_heavy = 1,
    df = df,
    corr_matrix = ot.CorrelationMatrix(2, (1.0, 0.75, 0.75, 1.0))
)
target = target.to(device)

# Estimate entropy as lower bound for flow NLLs
x = target.sample(64000).to(device)
log_p = target.log_prob(x).sum().item()
average_nats_per_dim = -log_p / (x.shape[0] * target.dim)
print(f"average_nats_per_dim: {average_nats_per_dim:.3f}")

# Set base distribuiton
q0 = nf.distributions.DiagGaussian(2, trainable=False)
q0 = q0.to(device)

# Architectural parameters for flow models
num_layers = 3
num_blocks = 2
num_hidden_channels = 64

# Parameters for plots
x_low, x_high = -5.0, 5.0
y_low, y_high = -5.0, 5.0

range_hist2d = ((x_low,x_high), (y_low, y_high))
extent_imshow = (x_low, x_high, y_low, y_high)

# Prepare grid
bins = 500
grid_size = 300
grid = torch.meshgrid(torch.linspace(x_low, x_high, grid_size), torch.linspace(y_low, y_high, grid_size), indexing='xy')
grid = torch.stack([grid[0].flatten(), grid[1].flatten()], dim=1).to(device)
print("grid.shape: ", grid.shape)

### Visualize target distribution:

In [ ]:
# Sample from target
x_np = target.sample(32000).detach().cpu().numpy()

# Compute likelihood on grid
target_log_p = target.log_prob(grid)
target_likelihood = torch.exp(target_log_p).reshape(grid_size, grid_size)
target_likelihood = target_likelihood.detach().cpu().numpy()

# Create plots
fig, axs = plt.subplots(1, 2, figsize=(6, 3))
axs[0].hist2d(x_np[:, 0], x_np[:, 1], bins=bins, range=range_hist2d, density=True)
axs[0].set_title('Target distribution samples')
axs[1].imshow(target_likelihood, extent=extent_imshow, aspect='auto', origin='lower', cmap='viridis')
axs[1].set_title('Target distribution likelihood')

fig.tight_layout()
fig.show()

### Fit NF models
We train discrete Neural Spline Flow (NSF) models here:

In [ ]:
# Prepare flow layers for NSF models
def build_flows():
    flows = []
    for i in range(num_layers):
        flows += [nf.flows.AutoregressiveRationalQuadraticSpline(2, num_blocks, num_hidden_channels)]
        flows += [nf.flows.LULinearPermute(2)]
    return flows


# Training algorithm
def train(nf_model: nf.NormalizingFlow, target, num_steps: int = 1000, batch_size: int = 512, show_every: Optional[int] = None, lr: float = 1e-3, weight_decay: float = 1e-5):
    """ Train the model on target distribution.

    Args:
        nf_model (nf.NormalizingFlow): NF model to train.
        target: Target distribution to train the model on.
        num_steps (int, optional): Number of training steps. Defaults to 1000.
        batch_size (int, optional): Defaults to 2**9.
        show_every (int, optional): Period for visualizing intermediate results (only for 2dim). If None, no intermediate results are computed. Defaults to None.
        lr (float, optional): Learning rate for optimizer. Defaults to 1e-3.
        weight_decay (float, optional): Weight decay for optimizer. Defaults to 1e-5.
    """

    # Prepare loss history
    loss_hist = np.array([])

    # Get training samples for initial loss
    x = target.sample(batch_size).float().to(device)
    # Compute and log initial loss
    loss = nf_model.forward_kld(x)
    loss_hist = np.append(loss_hist, loss.to('cpu').item())

    # Training loop
    optimizer = torch.optim.Adam(nf_model.parameters(), lr=lr, weight_decay=weight_decay)
    for step in tqdm(range(num_steps), desc="Training Loop"):
        optimizer.zero_grad()
        
        # Get training batch
        x = target.sample(batch_size).to(dtype=DEFAULT_DTYPE, device=device)
        
        # Compute and log loss
        loss = nf_model.forward_kld(x)
        loss_hist = np.append(loss_hist, loss.to('cpu').item())
        
        # Do backprop and optimizer step
        if ~(torch.isnan(loss) | torch.isinf(loss)):
            loss.backward()
            optimizer.step()
        
        # Compute and show intermediate results
        if show_every is not None and step % show_every == 0:
            nf_model.eval()
            # Sample from model
            samples, _ = nf_model.sample(16000)
            samples = samples.detach().cpu().numpy()
            # Compute likelihood on grid
            log_prob = nf_model.log_prob(grid)
            prob = torch.exp(log_prob).reshape(grid_size, grid_size)
            prob[torch.isnan(prob)] = 0
            prob = prob.detach().cpu().numpy()
            nf_model.train()

            # Visualize flow distribution
            fig, axs = plt.subplots(1, 2, figsize=(6, 3))
            axs[0].hist2d(samples[:, 0], samples[:, 1], bins=bins, range=range_hist2d, density=True)
            axs[0].set_title("Flow samples")
            axs[1].imshow(prob, extent=extent_imshow, aspect='auto', origin='lower', cmap='viridis')
            axs[1].set_title("Flow likelihood")
            tqdm.write(f"Current training loss: {loss.to('cpu').data.numpy():.3f}")
            plt.tight_layout()
            plt.show()

            # Print TTF params
            last_layer = nf_model.flows[-1]
            if isinstance(last_layer, (TailAffineMarginalTransform, ModifiedTailAffineMarginalTransform)):
                tqdm.write("pos_tail: ", last_layer.pos_tail)
                tqdm.write("neg_tail: ", last_layer.neg_tail)
                tqdm.write("shift: ", last_layer.shift)
                tqdm.write("scale: ", last_layer.scale)
                if isinstance(last_layer, ModifiedTailAffineMarginalTransform) and last_layer.mod in ["erfi", "lin"]:
                    if last_layer.a_pos.requires_grad:
                        print("a_pos: ", last_layer.a_pos)
                    if last_layer.a_neg.requires_grad:
                        print("a_neg: ", last_layer.a_neg)
                

    # Plot loss history
    fig, ax = plt.subplots(1, 1, figsize=(6, 3))
    ax.plot(loss_hist, label='loss')
    ax.set_ylim(0.0, max(loss_hist[0]*1.2, 10.0))
    ax.set_title('Training loss')
    plt.tight_layout()
    plt.show()

    # Plot final distribution
    nf_model.eval()
    samples, _ = nf_model.sample(32000)
    samples = samples.detach().cpu().numpy()
    log_prob = nf_model.log_prob(grid)
    prob = torch.exp(log_prob).reshape(grid_size, grid_size)
    prob[torch.isnan(prob)] = 0
    prob = prob.detach().cpu().numpy()

    fig, axs = plt.subplots(1, 2, figsize=(6, 3))
    axs[0].hist2d(samples[:, 0], samples[:, 1], bins=bins, range=range_hist2d, density=True)
    axs[0].set_title('Learned distribution samples')
    axs[1].imshow(prob, extent=extent_imshow, aspect='auto', origin='lower', cmap='viridis')
    axs[1].set_title('Learned distribution likelihood')
    plt.tight_layout()
    plt.show()

    # Print final TTF params
    last_layer = nf_model.flows[-1]
    if isinstance(last_layer, (TailAffineMarginalTransform, ModifiedTailAffineMarginalTransform)):
        tqdm.write("pos_tail: ", last_layer.pos_tail)
        tqdm.write("neg_tail: ", last_layer.neg_tail)
        tqdm.write("shift: ", last_layer.shift)
        tqdm.write("scale: ", last_layer.scale)
        if isinstance(last_layer, ModifiedTailAffineMarginalTransform) and last_layer.mod in ["erfi", "lin"]:
            if last_layer.a_pos.requires_grad:
                print("a_pos: ", last_layer.a_pos)
            if last_layer.a_neg.requires_grad:
                print("a_neg: ", last_layer.a_neg)

    plt.close()


# NLL computation
def get_average_nll(nf_model: nf.NormalizingFlow, target, num_samps: int = 16000) -> float:
    """ Compute average NLL per target sample and dimension.

    Args:
        target: Target distribution. Must implement sample() method.
        num_samps (int, optional): Number of independent samples drawn from target distribution to compute average NLL on. Defaults to 16000.

    Returns:
        nll (float): Average NLL per sample and dimension.
    """
    x1 = target.sample(num_samps)
    total_log_p = nf_model.log_prob(x1.to(dtype=DEFAULT_DTYPE, device=device)).sum().item()

    # Average total NLL over samples and dimension
    nll = -total_log_p / (num_samps * x1.shape[-1])
    return nll


# Count samples in bin
def bin_count(nf_model: nf.NormalizingFlow, target, dim: int, bin: Optional[tuple] = None, num_samps: int = 32000, repeats: int = 8) -> tuple[tuple[float, float], tuple[float, float]]:
    """
    Compute average number of true samples and flow samples in bin for specified dimension.

    Args:
        nf_model (nf.NormalizingFlow): NF model.
        target: Target distribution. Must implement sample() method.
        dim (int): Dimension in which to count.
        bin (tuple[float, float], optional): Interval (bin) in which to count samples. If given None, defaults to small interval around shift of final tail transform. Defaults to None.
        num_samps (int, optional): Number of independent samples drawn from both target and flow distribution. Defaults to 16000.
        repeats (int, optional): Number of repeats to average results over. Defaults to 8.

    Returns:
        (avg_num_true_in_bin, std_num_true_in_bin), (avg_num_flow_in_bin, std_num_flow_in_bin) (tuple[tuple[float, float], tuple[float, float]]): Results for target and flow samples (mean, std).
    """
    if bin is None:
        # Set default bin around shift of final transformation
        last_layer = nf_model.flows[-1]
        if isinstance(last_layer, (TailAffineMarginalTransform, ModifiedTailAffineMarginalTransform)):
            mu = last_layer.shift[dim].to(DEFAULT_DTYPE).item()
            bin = (mu - 0.1, mu + 0.1)
        else:
            bin = (-0.1, 0.1)

    print(f"bin ({bin[0]:.3f}, {bin[1]:.3f}) for dim {dim}:")

    # compute average number of samples in bin for true samples and flow samples
    results_true = []
    results_flow = []
    for i in tqdm(range(repeats), desc="Repeat"):
        true_samps = target.sample(num_samps)
        flow_samps, _ = nf_model.sample(num_samps)
        true_samps = true_samps[:, dim]
        flow_samps = flow_samps[:, dim]
        num_true_in_bin = ((bin[0] < true_samps) & (true_samps < bin[1])).sum().item()
        num_flow_in_bin = ((bin[0] < flow_samps) & (flow_samps < bin[1])).sum().item()
        results_true.append(num_true_in_bin)
        results_flow.append(num_flow_in_bin)

    avg_num_true_in_bin = float(np.mean(results_true))
    std_num_true_in_bin = float(np.std(results_true))
    avg_num_flow_in_bin = float(np.mean(results_flow))
    std_num_flow_in_bin = float(np.std(results_flow))

    return (avg_num_true_in_bin, std_num_true_in_bin), (avg_num_flow_in_bin, std_num_flow_in_bin)

### Baseline (NSF without tail trafo):

In [ ]:
# Construct flow model
nfm_baseline = nf.NormalizingFlow(q0=q0, flows=build_flows())
nfm_baseline = nfm_baseline.to(device)

# Print number of parameters
num_params_baseline = sum(p.numel() for p in nfm_baseline.parameters() if p.requires_grad)
print(f"Num params nfm_baseline: ", num_params_baseline)

In [ ]:
# Train model
train(nfm_baseline, target, 2000, show_every=400)

if device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# Compute average NLL per sample and dim
nll_baseline = get_average_nll(nfm_baseline, target, 64000)
print(f"Average NLL: {nll_baseline:.3f}")

# Count samples in bin around 0
(avg_true_samps_in_bin_dim0, std_true_samps_in_bin_dim0), (avg_flow_samps_in_bin_dim0, std_flow_samps_in_bin_dim0) = bin_count(nf_model=nfm_baseline, target=target, dim=0)
print(f"Average num true samps, num flow samps in bin (dim 0): {avg_true_samps_in_bin_dim0:.1f} ({std_true_samps_in_bin_dim0:.1f}), {avg_flow_samps_in_bin_dim0:.1f} ({std_flow_samps_in_bin_dim0:.1f})")
(avg_true_samps_in_bin_dim1, std_true_samps_in_bin_dim1), (avg_flow_samps_in_bin_dim1, std_flow_samps_in_bin_dim1) = bin_count(nf_model=nfm_baseline, target=target, dim=1)
print(f"Average num true samps, num flow samps in bin (dim 1): {avg_true_samps_in_bin_dim1:.1f} ({std_true_samps_in_bin_dim1:.1f}), {avg_flow_samps_in_bin_dim1:.1f} ({std_flow_samps_in_bin_dim1:.1f})")

if device.type == 'cuda':
    torch.cuda.empty_cache()

### TTF models

In [ ]:
# Tailparams
pos_tail_init = torch.tensor([0.0, 1/df])
neg_tail_init = torch.tensor([0.0, 1/df])

# Shift and scale
shift_init = torch.tensor([0.0, 0.0])
scale_init = torch.tensor([1.0, 1.0])

# Standard TTF
ttf_layer = ModifiedTailAffineMarginalTransform(
    features = 2,
    pos_tail_init = pos_tail_init.detach().clone(),
    neg_tail_init = neg_tail_init.detach().clone(),
    shift_init = shift_init.detach().clone(),
    scale_init = scale_init.detach().clone(),
    hd_only = False,
    mod = "std",
    fix_params = False,
)

# Construct flow model
nfm_ttf = nf.NormalizingFlow(q0=q0, flows=build_flows() + [ttf_layer])
nfm_ttf = nfm_ttf.to(device)

# Print number of parameters
num_params_ttf = sum(p.numel() for p in nfm_ttf.parameters() if p.requires_grad)
print(f"Num params nfm_ttf: ", num_params_ttf)

In [ ]:
# train model
train(nfm_ttf, target, 2000)

if device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# Compute average NLL per sample and dim
nll_ttf = get_average_nll(nfm_ttf, target, 32000)
print(f"Average NLL: {nll_ttf:.3f}")

# Count samples in bin around 0
(avg_true_samps_in_bin_dim0, std_true_samps_in_bin_dim0), (avg_flow_samps_in_bin_dim0, std_flow_samps_in_bin_dim0) = bin_count(nf_model=nfm_ttf, target=target, dim=0)
print(f"Average num true samps, num flow samps in bin (dim 0): {avg_true_samps_in_bin_dim0:.1f} ({std_true_samps_in_bin_dim0:.1f}), {avg_flow_samps_in_bin_dim0:.1f} ({std_flow_samps_in_bin_dim0:.1f})")
(avg_true_samps_in_bin_dim1, std_true_samps_in_bin_dim1), (avg_flow_samps_in_bin_dim1, std_flow_samps_in_bin_dim1) = bin_count(nf_model=nfm_ttf, target=target, dim=1)
print(f"Average num true samps, num flow samps in bin (dim 1): {avg_true_samps_in_bin_dim1:.1f} ({std_true_samps_in_bin_dim1:.1f}), {avg_flow_samps_in_bin_dim1:.1f} ({std_flow_samps_in_bin_dim1:.1f})")

if device.type == 'cuda':
    torch.cuda.empty_cache()

### Modified TTF versions

In [ ]:
# Tailparams
pos_tail_init = torch.tensor([0.0, 1/df])
neg_tail_init = torch.tensor([0.0, 1/df])

# Shift and scale
shift_init = torch.tensor([0.0, 0.0])
scale_init = torch.tensor([1.0, 1.0])

# TTF layer with modifications
ttf_mod_layer = ModifiedTailAffineMarginalTransform(
    features = 2,
    pos_tail_init = pos_tail_init.detach().clone(),
    neg_tail_init = neg_tail_init.detach().clone(),
    shift_init = shift_init.detach().clone(),
    scale_init = scale_init.detach().clone(),
    hd_only = False,
    mod = "qua",
    fix_params = False,
)

# Construct flow model
nfm_ttf_mod = nf.NormalizingFlow(q0=q0, flows=build_flows() + [ttf_mod_layer])
nfm_ttf_mod = nfm_ttf_mod.to(device)

# Print number of parameters
num_params_ttf_mod = sum(p.numel() for p in nfm_ttf_mod.parameters() if p.requires_grad)
print(f"Num params nfm_ttf_mod: ", num_params_ttf_mod)

In [ ]:
# Train model
train(nfm_ttf_mod, target, 2000, show_every=400)

if device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# Compute average NLL per sample and dim
nll_ttf_mod = get_average_nll(nfm_ttf_mod, target, 64000)
print(f"Average NLL: {nll_ttf_mod:.3f}")

# Count samples in bin around 0
(avg_true_samps_in_bin_dim0, std_true_samps_in_bin_dim0), (avg_flow_samps_in_bin_dim0, std_flow_samps_in_bin_dim0) = bin_count(nf_model=nfm_ttf_mod, target=target, dim=0)
print(f"Average num true samps, num flow samps in bin (dim 0): {avg_true_samps_in_bin_dim0:.1f} ({std_true_samps_in_bin_dim0:.1f}), {avg_flow_samps_in_bin_dim0:.1f} ({std_flow_samps_in_bin_dim0:.1f})")
(avg_true_samps_in_bin_dim1, std_true_samps_in_bin_dim1), (avg_flow_samps_in_bin_dim1, std_flow_samps_in_bin_dim1) = bin_count(nf_model=nfm_ttf_mod, target=target, dim=1)
print(f"Average num true samps, num flow samps in bin (dim 1): {avg_true_samps_in_bin_dim1:.1f} ({std_true_samps_in_bin_dim1:.1f}), {avg_flow_samps_in_bin_dim1:.1f} ({std_flow_samps_in_bin_dim1:.1f})")

if device.type == 'cuda':
    torch.cuda.empty_cache()

## Compare Flow Matching Models

### Define FM vector fields:

In [ ]:
import torch.nn as nn
from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import ODESolver
from flow_matching.utils import ModelWrapper
from torchdiffeq import odeint_adjoint


# Fourier time embedding
class TimeEmbedding(nn.Module):
    def __init__(self, time_emb_dim: int):
        super().__init__()
        self.time_emb_dim = time_emb_dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Embed 1-d time into {time_emb_dim}-dimensional space.

        Args:
            t (torch.Tensor): Shape [batch] or [batch, 1].

        Returns:
            temb (torch.Tensor): Shape [batch, time_emb_dim].
        """
        if t.dim() == 1:
            t = t[:, None]

        half_dim = self.time_emb_dim // 2
        max_freq = torch.log(torch.tensor(15.0, device=t.device))
        freqs = torch.rand(half_dim, device=t.device) * max_freq # random frequencies between 0 and max_freq
        args = t * freqs[None, :] * 2.0 * torch.pi
        temb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

        if self.time_emb_dim % 2 == 1:
            temb = torch.cat([temb, t], dim=-1)

        return temb


# Basic MLP
class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, num_blocks: int):
        super().__init__()

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
        )

        self.blocks = nn.Sequential()

        for _ in range(num_blocks):
            self.blocks.append(
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.blocks.append(
                nn.SiLU()
            )
        
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Shape [batch, self.input_dim]

        Returns:
            torch.Tensor: Shape [batch, self.output_dim]
        """
        x = self.input_layer(x)
        x = self.blocks(x)
        x = self.output_layer(x)
        return x


# Residual layer with smooth SiLU activation
class ResidualBlock(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Shape [batch, dim]

        Returns:
            torch.Tensor: Shape [batch, dim]
        """
        return x + self.net(x)


# Combine several residual layers to MLP
class ResidualMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, num_blocks: int):
        super().__init__()

        residual_blocks = [
            ResidualBlock(hidden_dim, hidden_dim)
            for _ in range(num_blocks)
        ]

        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
            *residual_blocks,
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor):
        return self.layers(x)


# Time-dependent vector field for Flow Matching models
class FMVectorField(nn.Module):
    """ Time-dependent vector field, modeled by a spatio-temporal residual MLP + Fourier time embedding. """
    def __init__(
        self,
        x_dim: int = 2,
        hidden_dim: int = 128,
        num_blocks: int = 6,
        time_emb_dim: int = 64,
    ):
        super().__init__()

        self.x_dim = x_dim
        self.time_embedding = TimeEmbedding(time_emb_dim)

        input_dim = x_dim + time_emb_dim

        self.net = ResidualMLP(input_dim, hidden_dim, x_dim, num_blocks)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Shape [batch, x_dim].
            t (torch.Tensor): Shape [batch] or [batch, 1].

        Returns:
            v (torch.Tensor): Velocity vector field at point x and time t (shape [batch, x_dim]).
        """
        t = t.to(dtype=x.dtype, device=x.device)

        # Ensure t has the same batch size as x
        if (t.dim() == 0) or (t.dim() == 1 and t.shape[0] == 1): # scalar
            t = t.expand(x.shape[0])
        
        temb = self.time_embedding(t)
        h = torch.cat([x, temb], dim=-1)

        return self.net(h)

### Alternative velocity field architectures with better NSF-like inductive bias

Add marginal vector fields:

In [ ]:
class MixtureWeightsMLP(nn.Module):
    """ Mixture weights for coord.wise vector field, learned by an spatio-temporal MLP with Fourier time-embedding. """
    def __init__(self, x_dim: int, hidden_dim: int, num_blocks: int, time_emb_dim: int):
        super().__init__()

        self.x_dim = x_dim
        self.time_embedding = TimeEmbedding(time_emb_dim)

        input_dim = x_dim + time_emb_dim

        self.net = MLP(input_dim, hidden_dim, x_dim * 2, num_blocks)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """ Compute weights for global and coord.wise vector fields at x, t.

        Args:
            x (torch.Tensor): Shape [batch, x_dim].
            t (torch.Tensor): Shape [batch] or [batch, 1].

        Returns:
            weights (torch.Tensor): Shape [batch, x_dim, 2].
        """

        t = t.to(dtype=x.dtype, device=x.device)

        # Ensure t has the same batch size as x
        if (t.dim() == 0) or (t.dim() == 1 and t.shape[0] == 1): # scalar
            t = t.expand(x.shape[0])

        temb = self.time_embedding(t)
        h = torch.cat([x, temb], dim=-1)

        batch_size = x.shape[0]

        # Compute raw weights
        weights = self.net(h).reshape(batch_size, self.x_dim, 2)

        # Apply softmax in each spatial dimension
        softmax = nn.Softmax(dim=2)
        weights = softmax(weights)

        return weights


class CoordWiseVF(nn.Module):
    """ Implements a time-dependent vector field as the weighted sum of a global vector field
        and coord.wise vector field to compensate for TTF artifacts. """
    def __init__(self, x_dim: int, hidden_dim: int, num_blocks: int, time_emb_dim: int):
        super().__init__()

        self.x_dim = x_dim
        self.time_embedding = TimeEmbedding(time_emb_dim)

        input_dim = x_dim + time_emb_dim

        # Global net
        self.global_net = ResidualMLP(input_dim, hidden_dim, x_dim, num_blocks)

        # 1-dimensional marginal (coord.wise) vector fields as inductive bias to compensate for TTF artifacts
        self.coord_nets = nn.ModuleList([ # NOTE/TODO: Das ist noch etwas blöd weil jetzt die Anzahl der Netze mit x_dim skaliert! Lieber ein einziges Netz für alle marginals (wenn das überhaupt was bringt).
            ResidualMLP(1 + time_emb_dim, hidden_dim, 1, num_blocks)
            for _ in range(x_dim)
        ])

        # Mixture weights
        self.weights_net = MixtureWeightsMLP(
            x_dim = x_dim,
            hidden_dim = 16,
            num_blocks = 1,
            time_emb_dim = time_emb_dim,
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Shape [batch, x_dim].
            t (torch.Tensor): Shape [batch] or [batch, 1].

        Returns:
            v (torch.Tensor): Velocity vector field at point x and time t (shape [batch, x_dim])
                              consisting of a global part and coord.wise marginal parts.
        """
        t = t.to(dtype=x.dtype, device=x.device)

        # Ensure t has the same batch size as x
        if (t.dim() == 0) or (t.dim() == 1 and t.shape[0] == 1): # scalar
            t = t.expand(x.shape[0])

        temb = self.time_embedding(t)

        # Global part
        global_in = torch.cat([x, temb], dim=-1)
        v_global = self.global_net(global_in)

        # Coord.wise part
        coord_vs = []
        for i, net in enumerate(self.coord_nets):
            coord_in = torch.cat([x[:, i:i+1], temb], dim=-1) # Spatial input for i-th vector field is only x_i
            coord_vs.append(net(coord_in))
        v_coord = torch.cat(coord_vs, dim=-1)

        # Mixture weights
        weights = self.weights_net(x, t)

        return weights[:, :, 0] * v_global + weights[:, :, 1] * v_coord

### Define FM loss:

In [ ]:
# Conditional FM loss for linear Gaussian prob. paths
mse_loss = nn.MSELoss()
def cfm_loss(vf: Union[FMVectorField, CoordWiseVF], x1: torch.Tensor, q0: nf.distributions.BaseDistribution) -> torch.Tensor:
    """
    Conditional Flow Matching loss for linear (OT) Gaussian Probability Path.

    Args:
        vf (FMVectorField): FM vector field.
        x1 (torch.Tensor): Data batch (shape: [batch_size, features]).
        q0 (nf.distributions.BaseDistribution): Base distribution of the FM model (Usually N(0,I)).

    Returns:
        loss (torch.Tensor): loss on data batch (shape: []).
    """
    batch_size = x1.shape[0]
    device = x1.device

    # Instantiate OT probability path
    path = AffineProbPath(scheduler=CondOTScheduler())

    # sample from base distribution
    x0, _ = q0.forward(batch_size)
    x0 = x0.to(dtype=DEFAULT_DTYPE, device=device)

    # sample time uniformly
    t = torch.rand(batch_size, device=device)

    # Sample probability path
    path_sample = path.sample(x0, x1, t)

    # Conditional FM l2 loss
    loss = mse_loss(vf(path_sample.x_t, path_sample.t), path_sample.dx_t)

    return loss


# Simulation-based log-prob computation with odeint_adjoint for ML-based CNF training using the adjoint method
class CNFLogProbODE(nn.Module):
    def __init__(self, vf: Union[FMVectorField, CoordWiseVF]):
        """
        Instantiate reverse-time augmented CNF dynamics for adjoint method.

        Args:
            vf (FMVectorField or CoordWiseVF): Time-dependent CNF velocity vector field.
            
        """
        super().__init__()
        self.vf = vf
        self.eps = None # Placeholder for Hutchinson noise

    def set_noise(self, eps: torch.Tensor):
        """
        Set noise for Hutchinson's trace estimator.

        Args:
            eps (torch.Tensor): Same shape as 'x', typically sampled from standard-normal or Rademacher distribution.
        """
        self.eps = eps

    def divergence_hutchinson(self, x: torch.Tensor, v: torch.Tensor):
        """
        Estimates div_x v(t, x) using Hutchinson's estimator.

        Args:
            x (torch.Tensor): Shape [batch, dim], representing the current state.
            v (torch.Tensor): Shape [batch, dim], representing the velocity field value v_theta(t, x).

        Returns:
            div (torch.Tensor): Shape [batch] containing the estimated divergence for each batch element.
        """
        v_dot_eps = torch.sum(v * self.eps) # type: ignore
        grad_v = torch.autograd.grad(
            v_dot_eps,
            x,
            create_graph=True,
            retain_graph=True,
        )[0]
        return torch.sum(grad_v * self.eps, dim=1) # type: ignore

    def forward(self, t: torch.Tensor, state: tuple[torch.Tensor, torch.Tensor]):
        """
        Compute the augmented CNF dynamics.

        Args:
            t (torch.Tensor): Scalar tensor representing the current ODE time. Shape [batch] or [batch, 1].
            state (tuple[torch.Tensor, torch.Tensor]): Tuple (x, logp_delta), where x has shape [batch, dim] and logp_delta has shape [batch].

        Returns:
            (tuple): Tuple (dx_dt, dlogp_delta_dt) representing the derivatives of the augmented state w.r.t. time.
        """
        x, logp_delta = state
        del logp_delta  # logp_delta does not affect the dynamics directly

        # Important for odeint_adjoint:
        # its forward pass is run under torch.no_grad(), but divergence
        # estimation requires autograd with respect to x.
        with torch.enable_grad():
            x = x.detach().requires_grad_(True)
            v = self.vf(x, t)
            div_v = self.divergence_hutchinson(x, v)

        dx_dt = v
        dlogp_delta_dt = -div_v

        return dx_dt, dlogp_delta_dt


# CNF log_prob computation with adjoint method for backpropagating gradients through ODE solver
def cnf_log_prob_adjoint(x: torch.Tensor, vf: Union[FMVectorField, CoordWiseVF], log_p0: Callable[[torch.Tensor], torch.Tensor]):
    """ Computes CNF log-probabilities using backward ODE integration.

    Args:
        x (torch.Tensor): Shape [batch, dim], containing data samples.
        vf (FMVectorField or CoordWiseVF): Callable neural vector field v_theta(t, x) returning a tensor with the same shape as x.
        log_p0 (Callable[[torch.Tensor], torch.Tensor]): Callable mapping latent samples z of shape [batch_size, dim] to log-probabilities of shape [batch_size].

    Returns:
        log_p1 (torch.Tensor): Shape [batch_size], containing log p_1(x).
    """
    odefunc = CNFLogProbODE(vf)

    batch_size = x.shape[0]
    logp_delta = torch.zeros(batch_size, device=x.device, dtype=x.dtype)

    # Hutchinson noise should be fixed for one ODE solve.
    eps = torch.randn_like(x)
    odefunc.set_noise(eps)

    # Integrate backward: data x at t1 -> latent z at t0.
    t_span = torch.tensor([1.0, 0.0], device=x.device, dtype=x.dtype)

    z_t, logp_delta_t = odeint_adjoint( # type: ignore
        odefunc,
        (x, logp_delta),
        t_span,
        method="dopri5",
        atol=1e-4,
        rtol=1e-4,
    )

    z = z_t[-1]
    logp_delta = logp_delta_t[-1] # type: ignore

    logp0 = log_p0(z)

    logp1 = logp0 - logp_delta

    return logp1

### Add final TTF layer:

In [ ]:
class IdentityTransform(Transform):
    """ Identity function in feature space. """

    def __init__(self, features: int):
        super().__init__()
        self.features = features

    def forward(self, z: torch.Tensor, context=None) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            z (torch.Tensor): Shape [batch, features].

        Returns:
            z, lad (tuple[torch.Tensor, torch.Tensor]): lad (=0) is summed over dimension: Shape [batch].
        """
        lad = torch.zeros_like(z)
        return z, lad.sum(dim=-1)

    def inverse(self, x: torch.Tensor, context=None) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x (torch.Tensor): Shape [batch, features].

        Returns:
            x, lad (tuple[torch.Tensor, torch.Tensor]): lad (=0) is summed over dimension: Shape [batch].
        """
        lad = torch.zeros_like(x)
        return x, lad.sum(dim=-1)
    

class FM_TTF_model(torch.nn.Module):
    """ Implementation of a Flow Matching model together with an optional final tail (TTF) transformation. """

    def __init__(self, device: torch.device, vf: Union[FMVectorField, CoordWiseVF], q0: nf.distributions.BaseDistribution, tail_trafo: Optional[Transform] = None):
        """
        Build combined model consisting of a FM velocity field and a final tail transformation.

        Args:
            device (torch.device): Device.
            vf (FMVectorField): Flow Matching velocity field.
            q0 (nf.distributions.BaseDistribution): Base distribution of the FM model.
            tail_trafo (Transform, optional): Final tail transformation. If given None, the identity transformation is used. Defaults to None.
        """
        super().__init__()
        self.features = vf.x_dim
        self.device = device
        self.vf = vf.to(device)
        self.q0 = q0.to(device)
        if tail_trafo is None:
            tail_trafo = IdentityTransform(features=vf.x_dim) # Default: Identity Transformation
        self.tail_trafo = tail_trafo.to(device)

        assert self.features == self.tail_trafo.features, f"Mismatch in dimension of vf ({self.features}) and dimension of tail_trafo ({self.tail_trafo.features})."

        # Fix TTF params
        if isinstance(self.tail_trafo, (TailAffineMarginalTransform, ModifiedTailAffineMarginalTransform)):
            print("Currently, learning the TTF params is not possible for FM-TTF models. Therefore, TTF params will be fixed.")
            if isinstance(self.tail_trafo, ModifiedTailAffineMarginalTransform):
                self.tail_trafo.fix_all()
            elif isinstance(self.tail_trafo, TailAffineMarginalTransform):
                self.tail_trafo.fix_tails()
                print("Fixed tailparams.")
                self.tail_trafo.shift.requires_grad = False
                print("Fixed location params.")
                self.tail_trafo._unc_scale.requires_grad = False
                print("Fixed scale params.")

    @torch.no_grad()
    def sample(self, num_samps: int, method: str = "dopri5", step_size: Optional[float] = None, **ode_extras) -> torch.Tensor:
        """ Sample from model.

        Args:
            num_samps (int): Number of samples.
            method (str, optional): Method for ODE solver. Defaults to "dopri5".
            step_size (float, optional): Stepsize for ODE solver. Must be None for adaptive solvers. Defaults to None.
            **ode_extras: Additional config for ODE solver.

        Returns:
            x1 (torch.Tensor): Samples (shape: [num_samps, self.features])
        """
        tqdm.write(f"Generating {num_samps} model samples...")

        # Sample from base distribution
        x0 = self.q0.sample(num_samps)
        if isinstance(x0, tuple):
            x0 = x0[0] # take only samples, not log_prob
        x0 = x0.to(self.device)

        # Solve forward ODE
        solver = ODESolver(ModelWrapper(self.vf))
        time_grid = torch.tensor([0.0, 1.0], device=self.device, dtype=x0.dtype)
        y = solver.sample(x0, step_size=step_size, method=method, time_grid=time_grid, **ode_extras)

        # Apply tail transformation
        x1, _ = self.tail_trafo.forward(y)

        return x1

    @torch.no_grad()
    def log_prob(self, x1: torch.Tensor, method: str = "dopri5", batch_size: int = 2**14, exact_divergence: bool = False, step_size: Optional[float] = None, position: Optional[int] = None, leave: Optional[bool] = None, **ode_extras) -> torch.Tensor:
        """ Compute log_prob on data x1. Computation is split in batches to reduce peak memory usage.

        Args:
            x1 (torch.Tensor): Data batch (shape: [batch_size, self.features])
            method (str, optional): Method for ODE solver. Defaults to "dopri5".
            batch_size (int): Size of batches on which likelihood is computed simultaneously (to save memory). Defaults to 2**14.
            exact_divergence (bool, optional): Whether to use exact diversion or Hutchinson estimator in ODE solver. Defaults to False.
            step_size (float, optional): Stepsize for ODE solver. Must be None for adaptive solvers. Defaults to None.
            position (int, optional): Vertical line offset for tqdm progress bar (give position=1 when calling this function from within an outer tqdm loop for cleaner progress bars). Defaults to None.
            leave (bool, optional): If True, keeps all traces of the tqdm progress bar upon termination of iteration. If None, will leave only if position is 0. Defaults to None.
            **ode_extras: Additional config for ODE solver.

        Returns:
            log_p (torch.Tensor): log prob values for all data points (summed over feature dimension) (shape: [batch]).
        """
        x1 = x1.to(self.device)

        tqdm.write(f"Computing likelihood on tensor of shape {x1.shape}...")
        # Apply inverse tail transformation
        y, lad = self.tail_trafo.inverse(x1)
        y = y.to(self.device)
        lad = lad.to(self.device)
        num_samps = y.shape[0]

        # Solve reverse-time ODE in batches
        solver = ODESolver(ModelWrapper(self.vf))
        time_grid = torch.tensor([1.0, 0.0], device=self.device, dtype=y.dtype)

        log_p = torch.zeros_like(lad)

        for start in tqdm(range(0, num_samps, batch_size), desc="Compute log_p on batch", leave=leave, position=position):

            # Compute batch log_p
            end = min(start + batch_size, num_samps)
            batch = y[start:end]

            _ , log_p_batch = solver.compute_likelihood(batch, self.q0.log_prob, step_size=step_size, method=method, time_grid=time_grid, exact_divergence=exact_divergence, **ode_extras)
            log_p[start:end] = log_p_batch.to(self.device)

            # Delete batch results to free memory
            del _ , batch, log_p_batch
            if self.device.type == 'cuda':
                torch.cuda.empty_cache()

        # Change of variables formula
        log_p = log_p + lad

        return log_p

    @torch.no_grad()
    def get_average_nll(self, target, num_samps: int = 16000, method: str = 'dopri5', batch_size: int = 2**14, exact_divergence: bool = True, step_size: Optional[float] = None, **ode_extras) -> float:
        """ Compute average NLL per target sample and dimension. Computation is split in batches to reduce peak usage.

        Args:
            target: Target distribution. Must implement sample() method.
            num_samps (int, optional): Number of independent samples drawn from target distribution to compute average NLL on. Defaults to 16000.
            method (str, optional): Method for ODE sovler. Defaults to 'dopri5'.
            batch_size (int, optional): Size of batches on which likelihood is computed simultaneously. Defaults to 2**14.
            exact_divergence (bool, optional): Whether to use exact divergence or Hutchinson estimator in ODE solver. Defaults to True.
            step_size (float, optional): Stepsize for ODE solver. Must be None for adaptive solvers. Defaults to None.
            **ode_extras: Additional config for ODE solver.

        Returns:
            nll (float): Average NLL per sample and dimension.
        """
        # Sample independent target data points
        x1 = target.sample(num_samps).to(dtype=DEFAULT_DTYPE, device=self.device)

        log_p = self.log_prob(x1, method, batch_size, exact_divergence, step_size, **ode_extras)

        # Average total NLL over samples and dimension
        nll = -log_p.sum().item() / (num_samps * self.features)

        return nll

    def train(self, target, num_steps: int = 1000, batch_size: int = 512, show_every: Optional[int] = None, show_final: bool = True, lr: float = 1e-3, weight_decay: float = 1e-5, ml_steps: Optional[int] = None):
        """ Train the model on target distribution. Trains only the FM vector field, not the final transformation.

        Args:
            target: Target distribution to train the model on. Must implement sample() method.
            num_steps (int, optional): Number of training steps. Defaults to 1000.
            batch_size (int, optional): Defaults to 512.
            show_every (int, optional): Period for visualizing intermediate results (only for 2dim). If None (default), no intermediate results are computed.
            lr (float, optional): Learning rate for optimizer. Defaults to 1e-3.
            weight_decay (float, optional): Weight decay for optimizer. Defaults to 1e-5.
            ml_steps (int, optional): Number of simulation-based Maximum-Likelihood (ML) steps after CFM training. If None (default), no ML training steps are performed.
        """
        target = target.to(self.device)

        # Prepare loss history
        loss_hist = np.array([])

        # Training loop
        optimizer = torch.optim.Adam(self.vf.parameters(), lr=lr, weight_decay=weight_decay)
        for step in tqdm(range(num_steps), desc="Training Loop"):
            optimizer.zero_grad()

            # Get training batch
            x = target.sample(batch_size).to(dtype=DEFAULT_DTYPE, device=self.device)
            # Apply inverse tail transform
            y, _ = self.tail_trafo.inverse(x)
            y = y.to(self.device)

            # Compute and log loss
            loss = cfm_loss(self.vf, y, self.q0)
            loss_hist = np.append(loss_hist, loss.to('cpu').item())

            # Do backprop and optimizer step
            if ~(torch.isnan(loss) | torch.isinf(loss)):
                loss.backward()
                optimizer.step()

            # Compute and show intermediate results
            if show_every is not None and self.features == 2:
                if step % show_every == 0:
                    # Sample from model
                    samples = self.sample(16000)
                    samples = samples.detach().cpu().numpy()
                    # Compute likelihood on grid
                    log_prob = self.log_prob(grid, position=1)
                    prob = torch.exp(log_prob).reshape(grid_size, grid_size)
                    prob[torch.isnan(prob)] = 0
                    prob = prob.detach().cpu().numpy()

                    # Visualize flow distribution
                    fig, axs = plt.subplots(1, 2, figsize=(6, 3))
                    axs[0].hist2d(samples[:, 0], samples[:, 1], bins=bins, range=range_hist2d, density=True)
                    axs[0].set_title("Flow samples")
                    axs[1].imshow(prob, extent=extent_imshow, aspect='auto', origin='lower', cmap='viridis')
                    axs[1].set_title("Flow likelihood")
                    tqdm.write(f"Current training loss: {loss.to('cpu').data.numpy():.3f}")
                    plt.tight_layout()
                    plt.show()

        # Plot loss history
        fig, ax = plt.subplots(1, 1, figsize=(6, 3))
        ax.plot(loss_hist, label='loss')
        ax.set_ylim(0.0, max(loss_hist[0]*1.2, 10.0))
        ax.set_title('Training loss')
        plt.tight_layout()
        plt.show()

        if show_final and self.features == 2:
            # Sample from model
            samples = self.sample(16000)
            samples = samples.detach().cpu().numpy()
            # Compute likelihood on grid
            log_prob = self.log_prob(grid, position=1)
            prob = torch.exp(log_prob).reshape(grid_size, grid_size)
            prob[torch.isnan(prob)] = 0
            prob = prob.detach().cpu().numpy()

            # Visualize flow distribution
            fig, axs = plt.subplots(1, 2, figsize=(6, 3))
            axs[0].hist2d(samples[:, 0], samples[:, 1], bins=bins, range=range_hist2d, density=True)
            axs[0].set_title("Flow samples")
            axs[1].imshow(prob, extent=extent_imshow, aspect='auto', origin='lower', cmap='viridis')
            axs[1].set_title("Flow likelihood")
            tqdm.write(f"Current training loss: {loss.to('cpu').data.numpy():.3f}")
            plt.tight_layout()
            plt.show()

        if ml_steps is not None:
            # Simulation-based ML training

            print_every = ml_steps // 10
            # Prepare loss history
            loss_hist_ml = np.array([])

            # ML training loop
            optimizer_ml = torch.optim.Adam(self.vf.parameters(), lr=2*lr, weight_decay=0.5*weight_decay)
            for step in tqdm(range(ml_steps), desc="ML Training Loop"):
                optimizer_ml.zero_grad()

                # Get training batch
                x = target.sample(batch_size).to(dtype=DEFAULT_DTYPE, device=self.device)
                # Apply inverse tail transform
                y, _ = self.tail_trafo.inverse(x)
                y = y.to(self.device)

                # Compute log_p on training batch using adjoint odeint to enable backprop with adjoint method
                log_prob = cnf_log_prob_adjoint(y, self.vf, self.q0.log_prob)

                # Compute and log ML loss
                loss_ml = -log_prob.mean()
                loss_hist_ml = np.append(loss_hist_ml, loss.to('cpu').item())

                # Do backprop and optimizer step
                if ~(torch.isnan(loss_ml) | torch.isinf(loss_ml)):
                    loss_ml.backward()
                    optimizer_ml.step()

                # Print progress
                print_every = ml_steps // 10
                if step % print_every == 0:
                    print(f"Step: {step}, ML loss: {loss_ml}")

            # Plot ML loss history
            fig, ax = plt.subplots(1, 1, figsize=(6, 3))
            ax.plot(loss_hist_ml, label='ML loss')
            ax.set_ylim(0.0, max(loss_hist_ml[0]*1.2, 10.0))
            ax.set_title('Training ML loss')
            plt.tight_layout()
            plt.show()

        # Plot final distribution
        if show_final:
            if self.features == 2:
                samples = self.sample(32000)
                samples = samples.detach().cpu().numpy()
                log_prob = self.log_prob(grid, exact_divergence=True, position=1)
                prob = torch.exp(log_prob).reshape(grid_size, grid_size)
                prob[torch.isnan(prob)] = 0
                prob = prob.detach().cpu().numpy()

                fig, axs = plt.subplots(1, 2, figsize=(6, 3))
                axs[0].hist2d(samples[:, 0], samples[:, 1], bins=bins, range=range_hist2d, density=True)
                axs[0].set_title("Flow samples")
                axs[1].imshow(prob, extent=extent_imshow, aspect='auto', origin='lower', cmap='viridis')
                axs[1].set_title("Flow likelihood")
                plt.tight_layout()
                plt.show()

        plt.close()

    @torch.no_grad()
    def bin_count(self, target, dim: int, bin: Optional[tuple[float, float]] = None, num_samps: int = 16000, repeats: int = 8,
                  method: str = "dopri5", step_size: Optional[float] = None, **ode_extras) -> tuple[tuple[float, float], tuple[float, float]]:
        """
        Compute average number of true samples and flow samples in bin for specified dimension.

        Args:
            target: Target distribution. Must implement sample() method.
            dim (int): Dimension in which to count.
            bin (tuple[float, float], optional): Interval (bin) in which to count samples. If given None, defaults to small interval around shift of final tail transform. Defaults to None.
            num_samps (int, optional): Number of independent samples drawn from both target and flow distribution. Defaults to 16000.
            repeats (int, optional): Number of repeats to average results over. Defaults to 8.
            method (str, optional): Method for ODE solver. Defaults to "dopri5".
            step_size (float, optional): Stepsize for ODE solver. Must be None for adaptive solvers. Defaults to None.
            **ode_extras: Additional config for ODE solver.

        Returns:
            (avg_num_true_in_bin, std_num_true_in_bin), (avg_num_flow_in_bin, std_num_flow_in_bin) (tuple[tuple[float, float], tuple[float, float]]): Results for target and flow samples (mean, std).
        """
        assert dim >= 0 and dim <= self.features - 1, f"Invalid dimension {dim} for model with {self.features} features."

        if bin is None:
            # Set default bin around shift of final transformation
            mu = 0.0
            if hasattr(self.tail_trafo, 'shift'):
                mu = self.tail_trafo.shift[dim].to(DEFAULT_DTYPE).item() # type: ignore 
            bin = (mu - 0.1, mu + 0.1)

        print(f"bin ({bin[0]:.3f}, {bin[1]:.3f}) for dim {dim}:")

        # Compute average number of samples in bin for target and flow samples
        results_true = []
        results_flow = []
        for i in tqdm(range(repeats), desc="Repeat"):
            true_samps = target.sample(num_samps)
            flow_samps = self.sample(num_samps, method=method, step_size=step_size, **ode_extras)
            true_samps = true_samps[:, dim]
            flow_samps = flow_samps[:, dim]
            num_true_in_bin = ((bin[0] < true_samps) & (true_samps < bin[1])).sum().item()
            num_flow_in_bin = ((bin[0] < flow_samps) & (flow_samps < bin[1])).sum().item()
            results_true.append(num_true_in_bin)
            results_flow.append(num_flow_in_bin)

        avg_num_true_in_bin = float(np.mean(results_true))
        std_num_true_in_bin = float(np.std(results_true))
        avg_num_flow_in_bin = float(np.mean(results_flow))
        std_num_flow_in_bin = float(np.std(results_flow))

        return (avg_num_true_in_bin, std_num_true_in_bin), (avg_num_flow_in_bin, std_num_flow_in_bin)

    def visualize(self, num_samps: int = 32000, exact_divergence: bool = True):
        if self.features == 2:
            samples = self.sample(num_samps)
            samples = samples.detach().cpu().numpy()
            log_prob = self.log_prob(grid, exact_divergence=exact_divergence)
            prob = torch.exp(log_prob).reshape(grid_size, grid_size)
            prob[torch.isnan(prob)] = 0
            prob = prob.detach().cpu().numpy()

            fig, axs = plt.subplots(1, 2, figsize=(6, 3))
            axs[0].hist2d(samples[:, 0], samples[:, 1], bins=bins, range=range_hist2d, density=True)
            axs[0].set_title("Flow samples")
            axs[1].imshow(prob, extent=extent_imshow, aspect='auto', origin='lower', cmap='viridis')
            axs[1].set_title("Flow likelihood")
            plt.tight_layout()
            plt.show()


### FM baseline (without tail transform):

In [ ]:
fm_baseline_model = FM_TTF_model(
    device,
    FMVectorField(
        x_dim = 2,
        hidden_dim = 32,
        num_blocks = 8,
        time_emb_dim = 16,
    ),
    q0,
)

# Print number of parameters
num_params_fm_baseline = sum(p.numel() for p in fm_baseline_model.parameters() if p.requires_grad)
print(f"Num params fm_baseline: ", num_params_fm_baseline)

In [ ]:
# Train model
fm_baseline_model.train(target, 1000, batch_size=2048, lr=2e-4, weight_decay=5e-6, show_every=499, ml_steps=20)

if fm_baseline_model.device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# Compute average NLL per sample and dim
nll_fm_baseline = fm_baseline_model.get_average_nll(target, 16000)
print(f"Average NLL: {nll_fm_baseline:.3f}")

# Count samples in bin around 0
(avg_true_samps_in_bin_dim0, std_true_samps_in_bin_dim0), (avg_flow_samps_in_bin_dim0, std_flow_samps_in_bin_dim0) = fm_baseline_model.bin_count(target, 0)
print(f"Average num true samps, num flow samps in bin (dim 0): {avg_true_samps_in_bin_dim0:.1f} ({std_true_samps_in_bin_dim0:.1f}), {avg_flow_samps_in_bin_dim0:.1f} ({std_flow_samps_in_bin_dim0:.1f})")
(avg_true_samps_in_bin_dim1, std_true_samps_in_bin_dim1), (avg_flow_samps_in_bin_dim1, std_flow_samps_in_bin_dim1) = fm_baseline_model.bin_count(target, 1)
print(f"Average num true samps, num flow samps in bin (dim 1): {avg_true_samps_in_bin_dim1:.1f} ({std_true_samps_in_bin_dim1:.1f}), {avg_flow_samps_in_bin_dim1:.1f} ({std_flow_samps_in_bin_dim1:.1f})")

if device.type == 'cuda':
    torch.cuda.empty_cache()

<!-- ### FM + TTF model -->

### FM + TTF model

In [ ]:
# Tailparams
pos_tail_init = torch.tensor([0.0, 1/df])
neg_tail_init = torch.tensor([0.0, 1/df])

# Shift and scale
shift_init = torch.tensor([0.0, 0.0])
scale_init = torch.tensor([1.0, 1.0])

fm_ttf_model = FM_TTF_model(
    device,
    FMVectorField(
        x_dim = 2,
        hidden_dim = 32,
        num_blocks = 8,
        time_emb_dim = 16,
    ),
    q0,
    ModifiedTailAffineMarginalTransform(
        features = 2,
        pos_tail_init = pos_tail_init.detach().clone(),
        neg_tail_init = neg_tail_init.detach().clone(),
        shift_init = shift_init.detach().clone(),
        scale_init = scale_init.detach().clone(),
        hd_only = False,
        mod = "std",
        a_pos_init = None,
        a_neg_init = None,
        fix_params = True,
    ),
)

# Print number of parameters
num_params_fm_ttf = sum(p.numel() for p in fm_ttf_model.parameters() if p.requires_grad)
print(f"Num params fm_ttf: ", num_params_fm_ttf)

In [ ]:
# Train model
fm_ttf_model.train(target, 3000, batch_size=2048, lr=2e-4, weight_decay=5e-6, show_every=900, ml_steps=100)

if fm_ttf_model.device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# Compute average NLL per sample and dim
nll_fm_ttf = fm_ttf_model.get_average_nll(target, 16000)
print(f"Average NLL: {nll_fm_ttf:.3f}")

# Count samples in bin around 0
(avg_true_samps_in_bin_dim0, std_true_samps_in_bin_dim0), (avg_flow_samps_in_bin_dim0, std_flow_samps_in_bin_dim0) = fm_ttf_model.bin_count(target, 0)
print(f"Average num true samps, num flow samps in bin (dim 0): {avg_true_samps_in_bin_dim0:.1f} ({std_true_samps_in_bin_dim0:.1f}), {avg_flow_samps_in_bin_dim0:.1f} ({std_flow_samps_in_bin_dim0:.1f})")
(avg_true_samps_in_bin_dim1, std_true_samps_in_bin_dim1), (avg_flow_samps_in_bin_dim1, std_flow_samps_in_bin_dim1) = fm_ttf_model.bin_count(target, 1)
print(f"Average num true samps, num flow samps in bin (dim 1): {avg_true_samps_in_bin_dim1:.1f} ({std_true_samps_in_bin_dim1:.1f}), {avg_flow_samps_in_bin_dim1:.1f} ({std_flow_samps_in_bin_dim1:.1f})")

if fm_ttf_model.device.type == 'cuda':
    torch.cuda.empty_cache()

### FM + modified TTF

In [ ]:
# Tailparams
pos_tail_init = torch.tensor([0.0, 1/df])
neg_tail_init = torch.tensor([0.0, 1/df])

# Shift and scale
shift_init = torch.tensor([0.0, 0.0])
scale_init = torch.tensor([1.0, 1.0])

fm_ttf_mod_model = FM_TTF_model(
    device,
    FMVectorField(
        x_dim = 2,
        hidden_dim = 64,
        num_blocks = 8,
        time_emb_dim = 32,
    ),
    q0,
    ModifiedTailAffineMarginalTransform(
        features = 2,
        pos_tail_init = pos_tail_init.detach().clone(),
        neg_tail_init = neg_tail_init.detach().clone(),
        shift_init = shift_init.detach().clone(),
        scale_init = scale_init.detach().clone(),
        hd_only = False,
        mod = "qua",
        a_pos_init = None,
        a_neg_init = None,
        fix_params = False,
    ),
)

# Print number of parameters
num_params_fm_ttf_mod = sum(p.numel() for p in fm_ttf_mod_model.parameters() if p.requires_grad)
print(f"Num params fm_ttf_mod: ", num_params_fm_ttf_mod)

In [ ]:
# Train model
fm_ttf_mod_model.train(target, 8000, batch_size=2048, lr=2e-4, weight_decay=0.0, show_every=2000)

if fm_ttf_mod_model.device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# Compute average NLL per sample and dim
nll_fm_ttf_mod = fm_ttf_mod_model.get_average_nll(target, 64000)
print(f"Average NLL: {nll_fm_ttf_mod:.3f}")

# Count samples in bin around 0
(avg_true_samps_in_bin_dim0, std_true_samps_in_bin_dim0), (avg_flow_samps_in_bin_dim0, std_flow_samps_in_bin_dim0) = fm_ttf_mod_model.bin_count(target, 0)
print(f"Average num true samps, num flow samps in bin (dim 0): {avg_true_samps_in_bin_dim0:.1f} ({std_true_samps_in_bin_dim0:.1f}), {avg_flow_samps_in_bin_dim0:.1f} ({std_flow_samps_in_bin_dim0:.1f})")
(avg_true_samps_in_bin_dim1, std_true_samps_in_bin_dim1), (avg_flow_samps_in_bin_dim1, std_flow_samps_in_bin_dim1) = fm_ttf_mod_model.bin_count(target, 1)
print(f"Average num true samps, num flow samps in bin (dim 1): {avg_true_samps_in_bin_dim1:.1f} ({std_true_samps_in_bin_dim1:.1f}), {avg_flow_samps_in_bin_dim1:.1f} ({std_flow_samps_in_bin_dim1:.1f})")

if fm_ttf_mod_model.device.type == 'cuda':
    torch.cuda.empty_cache()

### FM with coord.wise vector field + TTF

In [ ]:
# Tailparams
pos_tail_init = torch.tensor([0.0, 1/df])
neg_tail_init = torch.tensor([0.0, 1/df])

# Shift and scale
shift_init = torch.tensor([0.0, 0.0])
scale_init = torch.tensor([1.0, 1.0])

fm_coord_vf_ttf_model = FM_TTF_model(
    device,
    CoordWiseVF(
        x_dim = 2,
        hidden_dim = 32,
        num_blocks = 8,
        time_emb_dim = 16,
    ),
    q0,
    ModifiedTailAffineMarginalTransform(
        features = 2,
        pos_tail_init = pos_tail_init.detach().clone(),
        neg_tail_init = neg_tail_init.detach().clone(),
        shift_init = shift_init.detach().clone(),
        scale_init = scale_init.detach().clone(),
        hd_only = False,
        mod = "std",
        a_pos_init = None,
        a_neg_init = None,
        fix_params = True,
    ),
)

# Print number of parameters
num_params_fm_coord_vf_ttf = sum(p.numel() for p in fm_coord_vf_ttf_model.parameters() if p.requires_grad)
print(f"Num params fm_coord_vf_ttf: ", num_params_fm_coord_vf_ttf)

In [ ]:
# Train model
fm_coord_vf_ttf_model.train(target, 5000, batch_size=2048, lr=2e-4, weight_decay=1e-5, show_every=900)

if fm_coord_vf_ttf_model.device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# Compute average NLL per sample and dim
nll_fm_coord_vf_ttf = fm_coord_vf_ttf_model.get_average_nll(target, 64000)
print(f"Average NLL: {nll_fm_coord_vf_ttf:.3f}")

# Count samples in bin around 0
(avg_true_samps_in_bin_dim0, std_true_samps_in_bin_dim0), (avg_flow_samps_in_bin_dim0, std_flow_samps_in_bin_dim0) = fm_coord_vf_ttf_model.bin_count(target, 0)
print(f"Average num true samps, num flow samps in bin (dim 0): {avg_true_samps_in_bin_dim0:.1f} ({std_true_samps_in_bin_dim0:.1f}), {avg_flow_samps_in_bin_dim0:.1f} ({std_flow_samps_in_bin_dim0:.1f})")
(avg_true_samps_in_bin_dim1, std_true_samps_in_bin_dim1), (avg_flow_samps_in_bin_dim1, std_flow_samps_in_bin_dim1) = fm_coord_vf_ttf_model.bin_count(target, 1)
print(f"Average num true samps, num flow samps in bin (dim 1): {avg_true_samps_in_bin_dim1:.1f} ({std_true_samps_in_bin_dim1:.1f}), {avg_flow_samps_in_bin_dim1:.1f} ({std_flow_samps_in_bin_dim1:.1f})")

if fm_coord_vf_ttf_model.device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
torch.cuda.empty_cache()